# Kitchen Dishwasher Environment — RL with PPO

This notebook implements a complete reinforcement learning pipeline for a **custom kitchen environment**. The agent’s task is to clean dirty dishes by:

1. **Navigating** to dirty dishes on counters using cardinal movement (N/S/E/W)
2. **Picking up** dirty dishes
3. **Rinsing** them at the sink using the `use_sink` action
4. **Loading** them into the dishwasher using the `load_dishwasher` action

The environment is built from scratch on `gymnasium.Env` with a 9-action discrete action space,
procedurally generated kitchen layouts, and train/val/test seed splits.

We use **Proximal Policy Optimization (PPO)** from Stable-Baselines3 with a custom CNN feature extractor.

### Outline
1. Imports & Configuration
2. Custom Kitchen Environment Definition
3. Procedural Generation & Seed Splits
4. CNN Feature Extractor
5. Environment Factory
6. Training
7. Evaluation
8. Video Recording

## 1. Imports & Configuration

Key libraries:

- **`gymnasium`** — standard RL environment interface with `spaces` for defining observation/action spaces
- **`stable_baselines3`** — high-quality PPO implementation
- **`torch`** — PyTorch for the custom CNN feature extractor
- **`PIL`** — for rendering grid frames as RGB images
- **`imageio`** — for saving rollout frames as video files

No external grid library (e.g., MiniGrid) is used — the environment is built directly on `gymnasium.Env`.

In [ ]:
import gymnasium as gym
from gymnasium import spaces

import torch as th
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

import imageio
from PIL import Image, ImageDraw
from pathlib import Path
import numpy as np
import random
from typing import List, Tuple, Optional
from enum import IntEnum
from collections import deque

CURRENT_DIR = Path(".")
ENV_ID = "Kitchen-Dishwash-v0"
SEED = 42

## 2. Custom Kitchen Environment

The `KitchenEnv` is a fully custom `gymnasium.Env` representing a kitchen where an agent must
collect dirty dishes, rinse them at the sink, and load them into the dishwasher.

### Grid Layout (procedurally generated)
- **8×8 grid** (configurable) with wall borders
- **Sink** placed randomly in the top-left quadrant
- **Dishwasher** placed randomly in the bottom-right quadrant
- **1–2 dirty dishes** placed on random empty cells (kept small for faster learning)
- **0–1 counters** placed randomly; **no obstacles** (removed to simplify the task)
- **Agent** starts at a random empty cell
- Layout is validated via BFS to ensure the sink and dishwasher are reachable

### Cell Types

| Value | Symbol | Color |
|---|---|---|
| 0 | Empty | Dark gray |
| 1 | Wall | Gray |
| 2 | Counter | Brown |
| 3 | Sink | Blue |
| 4 | Dishwasher | Green |
| 6 | DirtyDish | Red |
| 7 | CleanDish | Light blue |

### Action Space (Discrete, 9 actions)

| Index | Name | Effect |
|---|---|---|
| 0 | move_n | Move one cell north (row-1) |
| 1 | move_s | Move one cell south (row+1) |
| 2 | move_e | Move one cell east (col+1) |
| 3 | move_w | Move one cell west (col-1) |
| 4 | pick_up | Pick up a dish at current cell |
| 5 | put_down | Put down carried dish at current cell |
| 6 | use_sink | Rinse carried dish (must be at sink) |
| 7 | load_dishwasher | Load clean dish (must be at dishwasher) |
| 8 | noop | Do nothing |

### Reward Structure

| Event | Reward |
|---|---|
| Per timestep | **-0.01** |
| Distance shaping (dense) | **+0.01 × 1/(1+dist)** toward next sub-goal |
| Picking up a dirty dish | **+0.2** |
| Rinse a dirty dish at the sink | **+0.5** |
| Load a clean dish into dishwasher | **+1.0** |
| Break a dish (put down on counter) | **-2.0** |
| All dishes cleaned bonus | **+2.0** |

The **distance shaping** provides dense reward every step, guiding the agent toward the nearest
dirty dish (if empty-handed), the sink (if carrying dirty), or the dishwasher (if carrying clean).

### Episode Termination
- All dishes are loaded into the dishwasher or broken (none remaining)
- Maximum steps reached (default 200)

In [ ]:
class CellType(IntEnum):
    EMPTY = 0
    WALL = 1
    COUNTER = 2
    SINK = 3
    DISHWASHER = 4
    OBSTACLE = 5
    DIRTY_DISH = 6
    CLEAN_DISH = 7


class KitchenAction(IntEnum):
    MOVE_N = 0
    MOVE_S = 1
    MOVE_E = 2
    MOVE_W = 3
    PICK_UP = 4
    PUT_DOWN = 5
    USE_SINK = 6
    LOAD_DISHWASHER = 7
    NOOP = 8


# Color map for rendering (RGB tuples per cell type)
CELL_COLORS = {
    CellType.EMPTY:      (40, 40, 40),
    CellType.WALL:       (100, 100, 100),
    CellType.COUNTER:    (139, 90, 43),
    CellType.SINK:       (0, 100, 255),
    CellType.DISHWASHER: (0, 180, 0),
    CellType.OBSTACLE:   (80, 0, 0),
    CellType.DIRTY_DISH: (255, 80, 80),
    CellType.CLEAN_DISH: (100, 200, 255),
}

AGENT_COLORS = {
    None:    (255, 255, 0),    # yellow  - not carrying
    'dirty': (255, 165, 0),    # orange  - carrying dirty dish
    'clean': (0, 255, 255),    # cyan    - carrying clean dish
}


class KitchenEnv(gym.Env):
    """
    Custom gymnasium environment: kitchen dish-cleaning task.

    The agent must pick up dirty dishes, rinse them at the sink,
    and load them into the dishwasher.
    """

    metadata = {"render_modes": ["rgb_array"], "render_fps": 6}

    def __init__(
        self,
        grid_size: int = 8,
        num_dishes_range: Tuple[int, int] = (1, 2),
        num_counters_range: Tuple[int, int] = (0, 1),
        num_obstacles_range: Tuple[int, int] = (0, 0),
        max_steps: int = 200,
        render_mode: str = "rgb_array",
    ):
        super().__init__()
        self.grid_size = grid_size
        self.num_dishes_range = num_dishes_range
        self.num_counters_range = num_counters_range
        self.num_obstacles_range = num_obstacles_range
        self.max_steps = max_steps
        self.render_mode = render_mode

        # Spaces
        self.action_space = spaces.Discrete(9)
        self.tile_px = 16
        img_size = grid_size * self.tile_px
        self.observation_space = spaces.Box(
            low=0, high=255,
            shape=(img_size, img_size, 3),
            dtype=np.uint8,
        )

        # State (initialized in reset)
        self.grid = None
        self.agent_pos = None
        self.carrying = None  # None, 'dirty', or 'clean'
        self.step_count = 0
        self.total_dishes = 0
        self.cleaned_count = 0
        self.broken_count = 0
        self.sink_pos = None
        self.dishwasher_pos = None

    # ------------------------------------------------------------------
    # Grid generation
    # ------------------------------------------------------------------

    def _random_empty_cell(self, rng):
        """Return a random interior cell that is currently EMPTY."""
        while True:
            r = rng.integers(1, self.grid_size - 1)
            c = rng.integers(1, self.grid_size - 1)
            if self.grid[r, c] == CellType.EMPTY:
                return (r, c)

    def _bfs_reachable(self, start, targets):
        """Check if all targets are reachable from start via BFS on walkable cells."""
        walkable = {
            CellType.EMPTY, CellType.SINK, CellType.DISHWASHER,
            CellType.DIRTY_DISH, CellType.CLEAN_DISH,
        }
        visited = set()
        queue = deque([start])
        visited.add(start)
        while queue:
            r, c = queue.popleft()
            for dr, dc in [(-1, 0), (1, 0), (0, 1), (0, -1)]:
                nr, nc = r + dr, c + dc
                if (nr, nc) not in visited and 0 <= nr < self.grid_size and 0 <= nc < self.grid_size:
                    if self.grid[nr, nc] in walkable:
                        visited.add((nr, nc))
                        queue.append((nr, nc))
        return all(t in visited for t in targets)

    def _generate_grid(self, rng):
        """Procedurally generate a kitchen layout. Retries until reachable."""
        for _ in range(200):  # retry limit
            grid = np.full((self.grid_size, self.grid_size), CellType.EMPTY, dtype=np.int8)

            # Walls on border
            grid[0, :] = CellType.WALL
            grid[-1, :] = CellType.WALL
            grid[:, 0] = CellType.WALL
            grid[:, -1] = CellType.WALL

            self.grid = grid  # needed by _random_empty_cell

            # Sink in top-left quadrant
            mid = self.grid_size // 2
            sr = rng.integers(1, max(2, mid))
            sc = rng.integers(1, max(2, mid))
            grid[sr, sc] = CellType.SINK
            self.sink_pos = (sr, sc)

            # Dishwasher in bottom-right quadrant
            dr = rng.integers(max(mid, 2), self.grid_size - 1)
            dc = rng.integers(max(mid, 2), self.grid_size - 1)
            if grid[dr, dc] != CellType.EMPTY:
                continue
            grid[dr, dc] = CellType.DISHWASHER
            self.dishwasher_pos = (dr, dc)

            # Counters
            n_counters = rng.integers(self.num_counters_range[0], self.num_counters_range[1] + 1)
            for _ in range(n_counters):
                pos = self._random_empty_cell(rng)
                grid[pos] = CellType.COUNTER

            # Obstacles (may be 0)
            n_obstacles = rng.integers(self.num_obstacles_range[0], self.num_obstacles_range[1] + 1)
            for _ in range(n_obstacles):
                pos = self._random_empty_cell(rng)
                grid[pos] = CellType.OBSTACLE

            # Dirty dishes
            n_dishes = rng.integers(self.num_dishes_range[0], self.num_dishes_range[1] + 1)
            dish_positions = []
            for _ in range(n_dishes):
                pos = self._random_empty_cell(rng)
                grid[pos] = CellType.DIRTY_DISH
                dish_positions.append(pos)

            # Agent
            agent_pos = self._random_empty_cell(rng)

            # Reachability check
            targets = [self.sink_pos, self.dishwasher_pos] + dish_positions
            if self._bfs_reachable(agent_pos, targets):
                self.grid = grid
                self.agent_pos = agent_pos
                self.total_dishes = n_dishes
                return

        # Fallback: minimal layout
        self.grid = np.full((self.grid_size, self.grid_size), CellType.EMPTY, dtype=np.int8)
        self.grid[0, :] = CellType.WALL
        self.grid[-1, :] = CellType.WALL
        self.grid[:, 0] = CellType.WALL
        self.grid[:, -1] = CellType.WALL
        self.grid[1, 1] = CellType.SINK
        self.sink_pos = (1, 1)
        self.grid[-2, -2] = CellType.DISHWASHER
        self.dishwasher_pos = (self.grid_size - 2, self.grid_size - 2)
        self.grid[1, 3] = CellType.DIRTY_DISH
        self.agent_pos = (1, 2)
        self.total_dishes = 1

    # ------------------------------------------------------------------
    # Observation rendering
    # ------------------------------------------------------------------

    def _get_obs(self):
        """Render the grid as a small RGB image for the CNN policy."""
        tp = self.tile_px
        img = np.zeros((self.grid_size * tp, self.grid_size * tp, 3), dtype=np.uint8)

        for r in range(self.grid_size):
            for c in range(self.grid_size):
                cell = int(self.grid[r, c])
                color = CELL_COLORS.get(CellType(cell), (40, 40, 40))
                img[r * tp:(r + 1) * tp, c * tp:(c + 1) * tp] = color

        # Draw agent — fill nearly the entire tile so CNN can see it clearly
        ar, ac = self.agent_pos
        agent_color = AGENT_COLORS[self.carrying]
        margin = 1  # 14x14 agent in 16x16 tile (was tp//4=4 -> tiny 8x8)
        img[ar * tp + margin:(ar + 1) * tp - margin,
            ac * tp + margin:(ac + 1) * tp - margin] = agent_color

        return img

    # ------------------------------------------------------------------
    # Reward shaping helpers
    # ------------------------------------------------------------------

    def _distance_reward_shaping(self):
        """Dense distance-based reward to guide the agent toward the next sub-goal."""
        ar, ac = self.agent_pos

        if self.carrying is None:
            # Not carrying anything -> move toward nearest dirty dish
            dish_cells = list(zip(*np.where(self.grid == CellType.DIRTY_DISH)))
            if dish_cells:
                min_dist = min(abs(ar - dr) + abs(ac - dc) for dr, dc in dish_cells)
                return 0.01 * (1.0 / (1.0 + min_dist))
            return 0.0
        elif self.carrying == 'dirty':
            # Carrying dirty dish -> move toward sink
            sr, sc = self.sink_pos
            dist = abs(ar - sr) + abs(ac - sc)
            return 0.01 * (1.0 / (1.0 + dist))
        elif self.carrying == 'clean':
            # Carrying clean dish -> move toward dishwasher
            dwr, dwc = self.dishwasher_pos
            dist = abs(ar - dwr) + abs(ac - dwc)
            return 0.01 * (1.0 / (1.0 + dist))
        return 0.0

    # ------------------------------------------------------------------
    # Core API
    # ------------------------------------------------------------------

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        rng = np.random.default_rng(seed)

        self.carrying = None
        self.step_count = 0
        self.cleaned_count = 0
        self.broken_count = 0

        self._generate_grid(rng)

        return self._get_obs(), self._get_info()

    def _get_info(self):
        return {
            "cleaned_count": self.cleaned_count,
            "broken_count": self.broken_count,
            "total_dishes": self.total_dishes,
            "carrying": self.carrying,
        }

    def _remaining_dishes(self):
        """Count dishes still on the grid + carried dish."""
        on_grid = int(np.isin(self.grid, [CellType.DIRTY_DISH, CellType.CLEAN_DISH]).sum())
        in_hand = 1 if self.carrying is not None else 0
        return on_grid + in_hand

    def step(self, action):
        action = int(action)
        reward = -0.01  # step penalty
        terminated = False
        truncated = False
        self.step_count += 1

        ar, ac = self.agent_pos

        # --- Movement ---
        if action <= KitchenAction.MOVE_W:
            deltas = {0: (-1, 0), 1: (1, 0), 2: (0, 1), 3: (0, -1)}
            dr, dc = deltas[action]
            nr, nc = ar + dr, ac + dc
            target = int(self.grid[nr, nc])
            walkable = {CellType.EMPTY, CellType.SINK, CellType.DISHWASHER,
                        CellType.DIRTY_DISH, CellType.CLEAN_DISH}
            if target in walkable:
                self.agent_pos = (nr, nc)

        # --- Pick up ---
        elif action == KitchenAction.PICK_UP:
            cell = int(self.grid[ar, ac])
            if self.carrying is None:
                if cell == CellType.DIRTY_DISH:
                    self.carrying = 'dirty'
                    self.grid[ar, ac] = CellType.EMPTY
                    reward += 0.2  # small reward for picking up
                elif cell == CellType.CLEAN_DISH:
                    self.carrying = 'clean'
                    self.grid[ar, ac] = CellType.EMPTY

        # --- Put down ---
        elif action == KitchenAction.PUT_DOWN:
            if self.carrying is not None:
                cell = int(self.grid[ar, ac])
                if cell in (CellType.COUNTER, CellType.OBSTACLE):
                    # Dish breaks!
                    reward += -2.0
                    self.broken_count += 1
                    self.carrying = None
                elif cell == CellType.EMPTY:
                    dish_type = CellType.DIRTY_DISH if self.carrying == 'dirty' else CellType.CLEAN_DISH
                    self.grid[ar, ac] = dish_type
                    self.carrying = None

        # --- Use sink ---
        elif action == KitchenAction.USE_SINK:
            if (ar, ac) == self.sink_pos and self.carrying == 'dirty':
                self.carrying = 'clean'
                reward += 0.5

        # --- Load dishwasher ---
        elif action == KitchenAction.LOAD_DISHWASHER:
            if (ar, ac) == self.dishwasher_pos and self.carrying == 'clean':
                self.carrying = None
                self.cleaned_count += 1
                reward += 1.0

        # --- Noop ---
        # action == KitchenAction.NOOP: do nothing

        # Dense distance-based reward shaping
        reward += self._distance_reward_shaping()

        # Check termination
        if self._remaining_dishes() == 0:
            terminated = True
            if self.cleaned_count > 0:
                reward += 2.0  # bonus for completing all dishes

        if self.step_count >= self.max_steps:
            truncated = True

        info = self._get_info()
        if terminated or truncated:
            info["episode"] = {
                "r": reward,
                "l": self.step_count,
            }

        return self._get_obs(), reward, terminated, truncated, info

    # ------------------------------------------------------------------
    # Rendering (for video recording)
    # ------------------------------------------------------------------

    def render(self):
        """Render a larger, human-readable RGB frame."""
        tile_px = 48
        size = self.grid_size * tile_px
        img = Image.new("RGB", (size, size), (30, 30, 30))
        draw = ImageDraw.Draw(img)

        labels = {
            CellType.WALL: "W",
            CellType.COUNTER: "Ctr",
            CellType.SINK: "Snk",
            CellType.DISHWASHER: "DW",
            CellType.OBSTACLE: "Obs",
            CellType.DIRTY_DISH: "D!",
            CellType.CLEAN_DISH: "D~",
        }

        for r in range(self.grid_size):
            for c in range(self.grid_size):
                cell = int(self.grid[r, c])
                color = CELL_COLORS.get(CellType(cell), (40, 40, 40))
                x0, y0 = c * tile_px, r * tile_px
                x1, y1 = x0 + tile_px - 1, y0 + tile_px - 1
                draw.rectangle([x0, y0, x1, y1], fill=color, outline=(60, 60, 60))
                label = labels.get(CellType(cell), "")
                if label:
                    draw.text((x0 + 4, y0 + 4), label, fill=(255, 255, 255))

        # Draw agent
        ar, ac = self.agent_pos
        agent_color = AGENT_COLORS[self.carrying]
        m = tile_px // 4
        ax0, ay0 = ac * tile_px + m, ar * tile_px + m
        ax1, ay1 = (ac + 1) * tile_px - m, (ar + 1) * tile_px - m
        draw.ellipse([ax0, ay0, ax1, ay1], fill=agent_color, outline=(255, 255, 255))
        carry_label = {None: "A", 'dirty': "A+D", 'clean': "A+C"}[self.carrying]
        draw.text((ax0 + 2, ay0 + 2), carry_label, fill=(0, 0, 0))

        return np.array(img)

## 3. Procedural Generation & Seed Splits

We split random seeds into **train / val / test** sets to evaluate generalization:
- **Train seeds (80)**: used during PPO training
- **Val seeds (10)**: used for evaluation during/after training
- **Test seeds (10)**: held out for final performance reporting

The `SeedCyclingWrapper` cycles through its seed list on each `reset()`, so the agent
sees a different procedurally-generated kitchen layout each episode.

In [ ]:
def get_seed_splits(base_seed=42, n_train=80, n_val=10, n_test=10):
    """Generate non-overlapping seed lists for train/val/test."""
    rng = np.random.RandomState(base_seed)
    all_seeds = rng.permutation(n_train + n_val + n_test)
    return {
        'train': all_seeds[:n_train].tolist(),
        'val': all_seeds[n_train:n_train + n_val].tolist(),
        'test': all_seeds[n_train + n_val:].tolist(),
    }


SEED_SPLITS = get_seed_splits(base_seed=SEED)
print(f"Train seeds: {len(SEED_SPLITS['train'])}")
print(f"Val seeds:   {len(SEED_SPLITS['val'])}")
print(f"Test seeds:  {len(SEED_SPLITS['test'])}")


class SeedCyclingWrapper(gym.Wrapper):
    """Cycles through a list of seeds on each reset for procedural generation."""

    def __init__(self, env, seeds):
        super().__init__(env)
        self.seeds = seeds
        self.seed_idx = 0

    def reset(self, **kwargs):
        kwargs['seed'] = self.seeds[self.seed_idx % len(self.seeds)]
        self.seed_idx += 1
        return self.env.reset(**kwargs)

## 4. CNN Feature Extractor

Observations are RGB images of the kitchen grid (128×128×3 for an 8×8 grid with 16px tiles).
We use a 3-layer CNN with **stride=2** convolutions to progressively reduce spatial dimensions:

```
Input Image (3, 128, 128)
    │
    ├── Conv2d(3 → 32, 3×3, stride=2, padding=1) + ReLU   → (32, 64, 64)
    ├── Conv2d(32 → 64, 3×3, stride=2, padding=1) + ReLU  → (64, 32, 32)
    ├── Conv2d(64 → 64, 3×3, stride=2, padding=1) + ReLU  → (64, 16, 16)
    ├── Flatten                                             → 16,384
    ├── Linear(16384 → 256) + ReLU
    └── Linear(256 → 128) + ReLU
         │
         └── 128-dim feature vector
```

The stride=2 convolutions reduce the spatial dimensions at each layer (128→64→32→16),
keeping the flattened size at ~16k instead of the ~1M that stride=1 would produce.
This avoids a massive information bottleneck and keeps the network trainable.

In [ ]:
class KitchenFeaturesExtractor(BaseFeaturesExtractor):
    """CNN feature extractor for kitchen grid RGB observations.

    Uses stride=2 convolutions to progressively reduce spatial dimensions:
    128x128 -> 64x64 -> 32x32 -> 16x16 -> flatten (16,384) -> 256 -> 128
    This avoids the massive 1M->128 bottleneck of stride=1 convolutions.
    """

    def __init__(self, observation_space, features_dim=128):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        with th.no_grad():
            sample = observation_space.sample()[None]
            sample = th.as_tensor(sample).float()
            n_flatten = self.cnn(sample).shape[1]
        self.linear = nn.Sequential(
            nn.Linear(n_flatten, 256),
            nn.ReLU(),
            nn.Linear(256, features_dim),
            nn.ReLU(),
        )

    def forward(self, observations):
        return self.linear(self.cnn(observations))


policy_kwargs = dict(
    features_extractor_class=KitchenFeaturesExtractor,
    features_extractor_kwargs=dict(features_dim=128),
)

## 5. Environment Factory

The `make_env` function returns a callable that creates a properly wrapped environment.

Wrapping chain:
1. `KitchenEnv(...)` → creates the raw environment (produces `(H, W, C)` RGB obs)
2. `SeedCyclingWrapper(env, seeds)` → cycles through seed list for procedural generation
3. `Monitor(env)` → tracks episode rewards and lengths
4. `DummyVecEnv([make_env()])` → SB3 vectorized env interface
5. `VecTransposeImage(vec_env)` → transposes `(H, W, C)` to `(C, H, W)` for PyTorch

In [ ]:
def make_env(split='train', grid_size=8, max_steps=200):
    """Returns a callable that creates a wrapped kitchen environment."""
    seeds = SEED_SPLITS.get(split, SEED_SPLITS['train'])

    def _init():
        env = KitchenEnv(
            grid_size=grid_size,
            num_dishes_range=(1, 2),
            num_counters_range=(0, 1),
            num_obstacles_range=(0, 0),
            max_steps=max_steps,
            render_mode="rgb_array",
        )
        env = SeedCyclingWrapper(env, seeds)
        env = Monitor(env)
        return env

    return _init

## 6. Training

We train a **PPO** agent on the custom kitchen environment.

### PPO Hyperparameters

| Parameter | Value | Description |
|---|---|---|
| `learning_rate` | 1e-3 | Adam optimizer learning rate (higher for faster learning with dense reward) |
| `n_steps` | 2048 | Steps per rollout before each update (more data per update) |
| `batch_size` | 64 | Minibatch size for gradient steps |
| `n_epochs` | 10 | Passes over the rollout buffer per update |
| `gamma` | 0.99 | Discount factor |
| `gae_lambda` | 0.95 | GAE smoothing factor |
| `clip_range` | 0.2 | PPO clipping range |
| `ent_coef` | 0.01 | Entropy bonus (low to allow policy specialization) |
| `vf_coef` | 0.5 | Value function loss coefficient |
| `max_grad_norm` | 0.5 | Gradient clipping |

### Custom Metrics Callback
A `DishMetricsCallback` logs kitchen-specific metrics (dishes cleaned, broken, success fraction)
to TensorBoard during training.

### Training Pipeline
1. Set random seeds for reproducibility
2. Create a vectorized environment with image transposition
3. Instantiate PPO with the custom CNN feature extractor
4. Train for 500,000 timesteps
5. Save the trained model to disk

In [ ]:
class DishMetricsCallback(BaseCallback):
    """Log kitchen-specific metrics during training."""

    def _on_step(self):
        infos = self.locals.get("infos", [])
        for info in infos:
            if "episode" in info:
                total = max(info.get("total_dishes", 1), 1)
                cleaned = info.get("cleaned_count", 0)
                broken = info.get("broken_count", 0)
                self.logger.record("kitchen/cleaned", cleaned)
                self.logger.record("kitchen/broken", broken)
                self.logger.record("kitchen/total_dishes", total)
                self.logger.record("kitchen/success_fraction", cleaned / total)
        return True


# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)

# Create vectorized environment
vec_env = DummyVecEnv([make_env(split='train')])
vec_env = VecTransposeImage(vec_env)

# Initialize PPO with custom CNN policy
model = PPO(
    "CnnPolicy",
    vec_env,
    policy_kwargs=policy_kwargs,
    learning_rate=1e-3,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    seed=SEED,
    verbose=1,
    tensorboard_log=str(CURRENT_DIR / "logs"),
)

# Train the agent
print("Starting training...")
model.learn(total_timesteps=500_000, callback=DishMetricsCallback(), progress_bar=True)

# Save the trained model
model_path = CURRENT_DIR / "model"
model.save(model_path)
print(f"Model saved to {model_path}.zip")

## 7. Evaluation

We evaluate the trained agent on the **validation seed split** with kitchen-specific metrics:

- **Mean episode reward** ± std
- **Fraction of dishes cleaned** (success metric)
- **Fraction of dishes broken** (failure metric)

Higher cleaned fraction and lower broken fraction indicate better performance.

In [ ]:
def evaluate_kitchen(model, seeds, n_episodes=20, grid_size=8, max_steps=200):
    """Evaluate with kitchen-specific success metrics."""
    all_rewards = []
    all_cleaned_fracs = []
    all_broken_fracs = []

    for i in range(n_episodes):
        env = KitchenEnv(
            grid_size=grid_size,
            max_steps=max_steps,
            render_mode="rgb_array",
        )
        obs, info = env.reset(seed=seeds[i % len(seeds)])
        done = False
        total_reward = 0.0

        while not done:
            obs_t = np.transpose(obs, (2, 0, 1))[np.newaxis]
            action, _ = model.predict(obs_t, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated

        total_dishes = max(env.total_dishes, 1)
        all_rewards.append(total_reward)
        all_cleaned_fracs.append(env.cleaned_count / total_dishes)
        all_broken_fracs.append(env.broken_count / total_dishes)
        env.close()

    print(f"Evaluation ({n_episodes} episodes on {len(seeds)} seeds):")
    print(f"  Mean reward:           {np.mean(all_rewards):.2f} \u00b1 {np.std(all_rewards):.2f}")
    print(f"  Mean cleaned fraction: {np.mean(all_cleaned_fracs):.2%}")
    print(f"  Mean broken fraction:  {np.mean(all_broken_fracs):.2%}")
    return all_rewards, all_cleaned_fracs, all_broken_fracs


val_rewards, val_cleaned, val_broken = evaluate_kitchen(
    model, SEED_SPLITS['val'], n_episodes=20
)

## 8. Video Recording

Record a video of the trained agent performing a single episode on a **test seed**.
Each frame is annotated with:
- Step count and action taken
- What the agent is carrying
- Dishes cleaned / broken counts
- Reward received

The video is saved as `demo.mp4` at 6 FPS.

In [ ]:
def annotate_frame(frame, env, step_idx, action_name=None, reward=None):
    """Add a status overlay to a rendered frame."""
    h, w = frame.shape[:2]
    # Create a wider canvas with a sidebar for text
    sidebar_w = 180
    canvas = np.full((h, w + sidebar_w, 3), 30, dtype=np.uint8)
    canvas[:, :w] = frame

    img = Image.fromarray(canvas)
    draw = ImageDraw.Draw(img)

    carry_str = {None: "nothing", 'dirty': "dirty dish", 'clean': "clean dish"}[env.carrying]
    lines = [
        f"Step: {step_idx}",
        f"Carrying: {carry_str}",
        f"Cleaned: {env.cleaned_count}/{env.total_dishes}",
        f"Broken: {env.broken_count}",
    ]
    if action_name:
        lines.append(f"Action: {action_name}")
    if reward is not None:
        lines.append(f"Reward: {reward:+.2f}")

    y = 10
    for line in lines:
        draw.text((w + 8, y), line, fill=(255, 255, 255))
        y += 18

    return np.array(img)


def record_kitchen_episode(model, seed, output_path="demo.mp4", max_steps=200, fps=6):
    """Record a single episode with annotated frames."""
    env = KitchenEnv(grid_size=8, max_steps=max_steps, render_mode="rgb_array")
    obs, info = env.reset(seed=seed)

    frames = []
    frame = env.render()
    frames.append(annotate_frame(frame, env, 0))

    done = False
    step_idx = 0
    while not done and step_idx < max_steps:
        obs_t = np.transpose(obs, (2, 0, 1))[np.newaxis]
        action, _ = model.predict(obs_t, deterministic=True)
        action_name = KitchenAction(int(action)).name

        obs, reward, terminated, truncated, info = env.step(action)
        step_idx += 1

        frame = env.render()
        frames.append(annotate_frame(frame, env, step_idx, action_name, reward))
        done = terminated or truncated

    output_path = str(CURRENT_DIR / output_path)
    imageio.mimsave(output_path, frames, fps=fps)
    print(f"Video saved to {output_path} ({len(frames)} frames, {step_idx} steps)")
    print(f"  Cleaned: {env.cleaned_count}/{env.total_dishes}, Broken: {env.broken_count}")
    env.close()


# Record on a held-out test seed
record_kitchen_episode(model, seed=SEED_SPLITS['test'][0])

---

### Next Steps

- **Increase grid size** to 10×10 or 12×12 for harder navigation tasks
- **Increase dish count** to (3, 6) for more complex multi-dish planning
- **Curriculum learning**: start with 1 dish and gradually increase
- **Recurrent policies**: try LSTM-based policies for better memory of dish states
- **Run test evaluation**: use `evaluate_kitchen(model, SEED_SPLITS['test'])` on held-out seeds
- **Monitor with TensorBoard**: run `tensorboard --logdir logs/` to track `kitchen/success_fraction`
- **Tune exploration**: increase `ent_coef` if the agent gets stuck; decrease if too random